# 1 · Generate the OpenFOAM training set (CPU runtime)

Runs the CFD cases listed in `cases.jsonl` and saves one `.npz` per finished case to Drive.

* **Runtime:** CPU, *High-RAM* (more cores). Check *Runtime → View resources* for the actual compute-unit rate.
* **Resumable:** finished cases are skipped. If the session dies, just run all cells again.
* **Parallel sessions:** open the notebook in more sessions of the same account and give each a different `SHARD` (0..`NUM_SHARDS`-1).
* Keep the browser tab open unless your plan has background execution.

In [ ]:
# --- Project location on Google Drive -------------------------------------
# Mounts Drive only if it is not mounted yet. NEVER run `rm -rf /content/drive`:
# if Drive is mounted it deletes your whole Google Drive.
import os, subprocess
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/windsurrogate'      # change if you like
PROJECT    = f'{DRIVE_ROOT}/idaho'                        # one folder per dataset (regions in config)
CODE_ZIP   = f'{DRIVE_ROOT}/windsurrogate_code.zip'       # upload the code zip here once
assert os.path.ismount('/content/drive'), 'Drive is not mounted'
assert os.path.exists(CODE_ZIP), f'missing {CODE_ZIP}: upload the code zip there first'
os.makedirs(PROJECT, exist_ok=True)

# --- Code ---------------------------------------------------------------------
if not os.path.exists('/content/windsurrogate'):
    subprocess.run(['unzip', '-q', '-o', CODE_ZIP, '-d', '/content'], check=True)
%cd /content/windsurrogate
!pip -q install rasterio pyproj pyyaml

In [ ]:
%%bash
# OpenFOAM v2412 (openfoam.com) from conda-forge, cached on Drive so later
# sessions only untar it (~1 min instead of ~5 min).
set -e
CACHE=/content/drive/MyDrive/windsurrogate/_cache/foam-env-2412.tar.gz
export MAMBA_ROOT_PREFIX=/content/mamba
if [ ! -x /content/bin/micromamba ]; then
  mkdir -p /content/bin
  curl -sSL -o /content/bin/micromamba https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64
  chmod +x /content/bin/micromamba
fi
if [ ! -d /content/mamba/envs/foam ]; then
  if [ -f "$CACHE" ]; then
    mkdir -p /content/mamba/envs && tar -xzf "$CACHE" -C /content/mamba/envs
  else
    /content/bin/micromamba create -y -q -n foam -c conda-forge openfoam=2412
    mkdir -p "$(dirname "$CACHE")" && tar -czf "$CACHE" -C /content/mamba/envs foam
  fi
fi
eval "$(/content/bin/micromamba shell hook -s bash)" && micromamba activate foam
simpleFoam -help | head -2 && echo "OpenFOAM $WM_PROJECT_VERSION ready"

In [ ]:
# Session config layered over config/default.yaml (edit region/geometry there or here).
import os
ncpu = os.cpu_count()
CONCURRENT = max(1, ncpu // 2)     # one serial case per physical core (vCPUs are hyper-threads)
cfg_text = f"""
project_dir: {PROJECT}
solver:
  concurrent_cases: {CONCURRENT}
  procs_per_case: 1
  activate: 'export MAMBA_ROOT_PREFIX=/content/mamba && eval "$(/content/bin/micromamba shell hook -s bash)" && micromamba activate foam'
output:
  work_dir: /content/work
"""
open('/content/colab.yaml', 'w').write(cfg_text)
print(cfg_text)
!nproc && free -g | head -2 && df -h /content | tail -1

## Prepare (only once per project)
Downloads USGS 3DEP 1 arc-second tiles for every region in `config/default.yaml` (Cervidae Peak and the Lost River Range), reprojects each to its UTM zone (`dem/<region>.tif`), and samples the cases with a spatial train/val/test split per region.
Run it in **one** session before starting other shards. Re-running is safe: existing files are reused.

In [ ]:
!python -m windsurrogate --config /content/colab.yaml prepare

## Optional: quick benchmark
Runs 1 case to measure minutes per case on this machine before committing hours.

In [ ]:
# !python -m windsurrogate --config /content/colab.yaml run --limit 1 --concurrent 1

## Progress
Counts finished cases per split and the status of failed ones. Safe to run at any time, in any session.

In [ ]:
import glob, json, collections
D = f'{PROJECT}/dataset'
for s in ['train', 'val', 'test']:
    print(f'{s:5s}: {len(glob.glob(f"{D}/{s}/*.npz"))}')
print('finished:', len(glob.glob(f'{D}/*/*.npz')), '| listed:', sum(1 for l in open(f'{PROJECT}/cases.jsonl') if l.strip()))
print('status files:', dict(collections.Counter(json.load(open(f)).get('status') for f in glob.glob(f'{D}/_status/*.json'))))

## Optional: retry failed cases
A case that fails twice is skipped. Cases interrupted by stopping a cell also count as failed.
Run this in **one** session, before starting the shards, to give them another try. It only deletes status files, never results.

In [ ]:
# !grep -l '"status": "failed"' {PROJECT}/dataset/_status/*.json | xargs -r rm

## Run this session's shard
One shard per server (*New Colab Server*, CPU High-RAM). With N servers use `NUM_SHARDS = N` and `SHARD = 0 .. N-1`, all at the same time.
`NUM_SHARDS` may change between batches, as long as no worker from the previous batch is still running.

In [ ]:
SHARD, NUM_SHARDS = 0, 2        # e.g. 2 servers -> (0, 2) and (1, 2)
HOURS = 11                      # stop starting new cases after this many hours (Colab sessions end after ~12 h)
!python -m windsurrogate --config /content/colab.yaml run --shard {SHARD} --num-shards {NUM_SHARDS} --hours {HOURS}

In [ ]:
!python -m windsurrogate --config /content/colab.yaml status

Quick look at a finished case (QA):

In [ ]:
import glob
from IPython.display import Image
f = sorted(glob.glob(f'{PROJECT}/dataset/train/*.npz'))[-1]
!python -m windsurrogate --config /content/colab.yaml plot-sample {f} /content/sample.png
Image('/content/sample.png')